# UCI Sticky-Sampler Sparsity Ablation — Investigation

Analyzes the output of `uci_sparsity_ablation.py`: for one `(dataset, split, hidden_variant)`,
a single MAP is fit, pruned to a target-sparsity-preserving threshold (RMSE-tolerance gated,
same shape as `fast_mnist_cnn.py::prune_x_ref`), refit in the active subspace, and both sticky
samplers (`grid_sticky_zigzag`, `grid_sticky_boomerang`) are cold-started from that pruned+refit
reference point (`cold_start_threshold=cold_start_mask`).

The core question: **does cold-starting from a MAP-informed sparse structure produce a
genuinely higher final sampler sparsity than the current unpruned-MAP baseline
(`uci_bnn_grid.py`, which always starts fully thawed)** — and if so, at what cost (RMSE,
gradient evals)?

Two source notebooks contributed sections here, kept selective rather than merged wholesale:
- `grid_uci_investigate.ipynb` (sections 1-6: metrics/sparsity-profile/noise on **resampled
  `samples`**) — metrics table and sparsity-profile plots reused directly; NUTS-specific and
  MAP-re-derivation cells dropped (not applicable — only 2 sticky samplers here, and this
  script's checkpoint already carries `rmse_map`/`rmse_pruned`/`rmse_refit` directly, no need
  to refit a second MAP just to sanity-check it).
- `grid_uci_investigate_skeletons.ipynb` (section 7: **raw skeleton diagnostics**) — event-type
  breakdown, `t_max`, `max_ratio`, and sparsity-dynamics (freeze/thaw trajectory) reused; the
  rate-reproduction section (7f) dropped since it needs `positions`/`velocities`/`times`, which
  this script's `save_skeleton` does not save (only `diagnostics`/`grid_t_max_log`/
  `frozen_mask_final`/resampled `samples` are kept, matching `uci_sparsity_ablation.py`'s
  leaner payload).

New here (in neither source notebook): the **prune/refit provenance** section — RMSE at
MAP/pruned/refit and the full threshold sweep curve, loaded from the saved
`<dataset>_split<k>_pruned_refit.pt` checkpoint — and the sparsity-dynamics section is
re-framed around the **cold-start value as an explicit reference line**, since that's the
number this whole ablation is testing against.

In [ ]:
from __future__ import annotations

import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import Tensor
from torch.distributions import Normal

plt.rcParams.update({
    "axes.spines.top":    False,
    "axes.spines.right":  False,
    "axes.grid":          True,
    "grid.alpha":         0.3,
    "font.size":          11,
})

import os
if Path.cwd().name == "notebooks":
    os.chdir("..")

## 1. Choose (dataset, split, hidden_variant) and load files

Two directory trees, matching `uci_sparsity_ablation.py`'s own layout:
- `OUT_MAPS_DIR` — one pruned+refit checkpoint per `(dataset, split)`, reused across runs
  (`<dataset>_split<k>_pruned_refit.pt`).
- `OUT_DIR` — one skeleton `.pt` per sampler, under `<dataset>/split_<k>/` (or
  `<variant>/<dataset>/split_<k>/` for a non-default `--hidden-variant`).

In [ ]:
DATASET        = "boston"
SPLIT_ID       = 0
HIDDEN_VARIANT = "small"          # matches --hidden-variant when the ablation script was run
DEFAULT_HIDDEN_VARIANT = "small"  # ablation script's own default -- flat layout iff this matches

OUT_MAPS_DIR = Path("results/maps/uci_sparsity_ablation")
OUT_DIR      = Path("results/grid/uci_sparsity_ablation")

_maps_root = OUT_MAPS_DIR if HIDDEN_VARIANT == DEFAULT_HIDDEN_VARIANT else OUT_MAPS_DIR / HIDDEN_VARIANT
_grid_root = OUT_DIR if HIDDEN_VARIANT == DEFAULT_HIDDEN_VARIANT else OUT_DIR / HIDDEN_VARIANT

ckpt_path  = _maps_root / f"{DATASET}_split{SPLIT_ID:02d}_pruned_refit.pt"
split_dir  = _grid_root / DATASET / f"split_{SPLIT_ID:02d}"

LABELS = {"grid_sticky_zigzag": "Sticky ZigZag", "grid_sticky_boomerang": "Sticky Boomerang"}
COLORS = {"grid_sticky_zigzag": "#1F77B4", "grid_sticky_boomerang": "#FA5C00"}
LINESTYLES = {"grid_sticky_zigzag": "-", "grid_sticky_boomerang": "-"}

print("checkpoint:", ckpt_path, "exists:", ckpt_path.exists())
print("skeletons :", split_dir)
skel_files = sorted(split_dir.glob("*_skeleton.pt")) if split_dir.exists() else []
for p in skel_files:
    print(" ", p.name)
assert ckpt_path.exists(), f"No pruned+refit checkpoint at {ckpt_path} -- run uci_sparsity_ablation.py first."
assert skel_files, f"No skeleton .pt files found in {split_dir}."


In [ ]:
ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)

runs: dict[str, dict] = {}
for p in skel_files:
    stem = p.stem.removesuffix("_skeleton")
    payload = torch.load(p, map_location="cpu", weights_only=False)
    diag_log = payload.get("diagnostics")
    runs[stem] = dict(
        payload = payload,
        diag_df = pd.DataFrame(diag_log) if diag_log else None,
        label   = LABELS.get(stem, stem),
        color   = COLORS.get(stem, "grey"),
        ls      = LINESTYLES.get(stem, "-"),
    )
    n_diag = len(diag_log) if diag_log else 0
    print(f"[{stem}] {payload['samples'].shape[0]} resampled draws, {n_diag} diagnostic rows, "
          f"D={payload['x_ref'].shape[0]}")

sampler_order = list(runs.keys())

## 2. Prune / refit provenance

Not in either source notebook -- specific to this ablation. `ckpt` is the saved
`<dataset>_split<k>_pruned_refit.pt`: one MAP, pruned once (threshold swept over
`np.logspace(-3, 0, ...)` candidates, picking the **largest sparsity whose RMSE stays within
`rmse_tolerance` of the unpruned baseline** -- see `prune_to_tolerance` in
`uci_sparsity_ablation.py`), then refit (masked-gradient Adam, frozen coordinates held at
exactly 0). Both sticky samplers below were cold-started from the SAME `x_ref`/`cold_start_mask`
saved here.

In [ ]:
print(f"D = {ckpt['x_ref'].shape[0]}")
print(f"achieved_sparsity = {ckpt['achieved_sparsity']:.4f}  "
      f"(rmse_tolerance = {ckpt['rmse_tolerance']:.4f})")
print()
print(f"RMSE   MAP    = {ckpt['rmse_map']:.4f}")
print(f"RMSE   pruned = {ckpt['rmse_pruned']:.4f}  (delta = {ckpt['rmse_pruned'] - ckpt['rmse_map']:+.4f})")
print(f"RMSE   refit  = {ckpt['rmse_refit']:.4f}  (delta = {ckpt['rmse_refit'] - ckpt['rmse_map']:+.4f})")
print()
print(f"||grad||_inf  MAP    = {ckpt['grad_norm_inf_map']:.4e}")
print(f"||grad||_inf  pruned = {ckpt['grad_norm_inf_pruned']:.4e}")
print(f"||grad||_inf  refit  = {ckpt['grad_norm_inf_refit']:.4e}")

In [ ]:
sweep = pd.DataFrame(ckpt["sweep_log"])
if sweep.empty:
    print("No sweep log (achieved_sparsity was likely 0 -- no pruning needed).")
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(sweep["multiplier"], sweep["achieved_frac"], color="#1F77B4", lw=1.5)
    axes[0].axhline(ckpt["achieved_sparsity"], color="black", ls=":", lw=1.0,
                     label=f"chosen ({ckpt['achieved_sparsity']:.3f})")
    axes[0].set_xscale("log")
    axes[0].set_xlabel("threshold multiplier (× per-coord prior std)")
    axes[0].set_ylabel("achieved sparsity (frac of freezable coords)")
    axes[0].set_title(f"{DATASET.capitalize()} — prune threshold sweep")
    axes[0].legend(fontsize=8)

    axes[1].plot(sweep["achieved_frac"], sweep["rmse"], color="#FA5C00", lw=1.5)
    axes[1].axhline(ckpt["rmse_map"] + ckpt["rmse_tolerance"], color="black", ls="--", lw=1.0,
                     label=f"tolerance ceiling ({ckpt['rmse_map'] + ckpt['rmse_tolerance']:.3f})")
    axes[1].axhline(ckpt["rmse_map"], color="grey", ls=":", lw=1.0, label=f"MAP RMSE ({ckpt['rmse_map']:.3f})")
    axes[1].axvline(ckpt["achieved_sparsity"], color="black", ls=":", lw=1.0)
    axes[1].set_xlabel("achieved sparsity")
    axes[1].set_ylabel("RMSE at that threshold (pre-refit)")
    axes[1].set_title(f"{DATASET.capitalize()} — sparsity vs. RMSE tradeoff")
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

## 3. Metrics table (resampled `samples`)

Adapted from `grid_uci_investigate.ipynb` section 4 -- same RMSE/NLL/CRPS/coverage/ESS
definitions, restricted to the two sticky samplers this script actually runs. `Grad evals/event`
and `t_max` columns fold in what that source notebook's section 4b kept separate (only 2
samplers here, no need for its own table).

In [ ]:
from sazz.gpu_friendly.scripts.uci_bnn_grid import (
    load_raw_datasets, make_split, BNNConfig, BASE_SEED, DTYPE, DEVICE,
)
from sazz.gpu_friendly.scripts.uci_sparsity_ablation import build_bm_only

raw = load_raw_datasets((DATASET,))
X_all, y_all = raw[DATASET]
data = make_split(X_all, y_all, seed=BASE_SEED + SPLIT_ID, dtype=DTYPE, device=DEVICE)
X_test, y_test, y_std = data["X_test"], data["y_test"], data["y_std"]

cfg = BNNConfig(
    layer_sizes=ckpt["layer_sizes"], activation=ckpt["activation"],
    prior_sigma_scale=ckpt["prior_sigma_scale"],
)
bm = build_bm_only(data, cfg)
print(f"D = {bm.D}   X_test: {tuple(X_test.shape)}   y_std: {y_std:.4f}")

@torch.no_grad()
def predict_all(bm, weight_samples: Tensor, X_new: Tensor) -> Tensor:
    return torch.stack([
        torch.func.functional_call(bm.module, bm.param_dict_fn(beta), (X_new,)).squeeze(-1)
        for beta in weight_samples
    ])  # [S, N]

for stem, s in runs.items():
    samples = s["payload"]["samples"].to(dtype=DTYPE)   # [S, D], last column is log_sigma
    weight_samples = samples[:, :-1]
    preds = predict_all(bm, weight_samples, X_test)
    s["samples"] = samples
    s["weight_samples"] = weight_samples
    s["mean_pred"] = preds.mean(0)
    s["epist_std"] = preds.std(0)
    noise_samples = samples[:, -1].exp()
    s["noise_samples"] = noise_samples
    s["noise_std_eff"] = float(noise_samples.mean())
    s["total_std"] = (s["epist_std"] ** 2 + s["noise_std_eff"] ** 2).sqrt()
    print(f"  [{stem}] predictions computed")

In [ ]:
from sazz.utils.metrics import ess_per_coord

def compute_rmse(y_true, mean_pred, y_std):
    return float(((mean_pred - y_true) ** 2).mean().sqrt()) * y_std

def compute_nll(y_true, mean_pred, total_std, y_std):
    ll = (-0.5 * ((y_true - mean_pred) / total_std) ** 2
          - total_std.log() - 0.5 * math.log(2 * math.pi)).mean()
    return float(-ll + math.log(y_std))

def compute_crps(y_true, mean_pred, total_std, y_std):
    d = Normal(0.0, 1.0)
    sigma = total_std * y_std
    z = (y_true * y_std - mean_pred * y_std) / sigma
    return float((sigma * (z * (2*d.cdf(z) - 1) + 2*d.log_prob(z).exp()
                           - 1/math.sqrt(math.pi))).mean())

def compute_coverage(y_true, mean_pred, total_std, level=0.9):
    z = Normal(0.0, 1.0).icdf(torch.tensor(0.5 + level / 2))
    return float(((y_true - mean_pred).abs() <= z * total_std).float().mean())

rows = []
for stem, s in runs.items():
    ess = ess_per_coord(s["weight_samples"])
    r = s["payload"]
    elapsed = r.get("elapsed_sec")
    grad_evals = r.get("gradient_evals")
    n_events = r.get("n_events")
    t_max_log = r.get("grid_t_max_log")
    rows.append({
        "Sampler":          s["label"],
        "RMSE":             compute_rmse(y_test, s["mean_pred"], y_std),
        "NLL":              compute_nll(y_test, s["mean_pred"], s["total_std"], y_std),
        "CRPS":             compute_crps(y_test, s["mean_pred"], s["total_std"], y_std),
        "Cov 90%":          compute_coverage(y_test, s["mean_pred"], s["total_std"], 0.90),
        "Cov 95%":          compute_coverage(y_test, s["mean_pred"], s["total_std"], 0.95),
        "ESS min":          float(ess.min()),
        "ESS/s":            float(ess.min()) / elapsed if elapsed else float("nan"),
        "ESS/grad":         (float(ess.min()) / grad_evals) if grad_evals else float("nan"),
        "Grad evals/event": (grad_evals / n_events) if (grad_evals and n_events) else float("nan"),
        "t_max mean":       float(np.mean(t_max_log)) if t_max_log else float("nan"),
        "Final sparsity":   float(r["frozen_mask_final"].float().mean()),
        "Time (s)":         elapsed if elapsed is not None else float("nan"),
    })

metrics_df = pd.DataFrame(rows).set_index("Sampler")

def highlight_coverage(col):
    target = 0.90 if "90" in col.name else 0.95
    best = (col - target).abs().idxmin()
    return ["background-color: #68dc0f" if idx == best else "" for idx in col.index]

display(
    metrics_df.style
    .highlight_min(subset=["RMSE", "NLL", "CRPS"], color="#68dc0f")
    .highlight_max(subset=["ESS/s", "ESS/grad", "Final sparsity"], color="#68dc0f")
    .apply(highlight_coverage, subset=["Cov 90%", "Cov 95%"])
    .format(precision=5, na_rep="n/a")
)

## 4. Sparsity profiles (resampled `samples`)

Adapted from `grid_uci_investigate.ipynb` section 5, unchanged in spirit: per-parameter
sparsity across the posterior draws, and model size (active-coordinate count) over the
resampled chain. The cold-start value (`achieved_sparsity` from the checkpoint) is marked as a
reference line on the right panel -- if the resampled-chain sparsity tracks near it, cold-starting
held; if it drifts well below, the chain thawed most of what pruning froze.

In [ ]:
FREEZE_THR = 1e-3
window = 100

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for stem, s in runs.items():
    ws = s["weight_samples"]  # [S, D]

    sparsity = (ws.abs() < FREEZE_THR).float().mean(dim=0).numpy()
    sorted_sparsity = np.sort(sparsity)
    axes[0].scatter(range(len(sorted_sparsity)), sorted_sparsity,
                     alpha=0.5, color=s["color"], label=s["label"], s=8)

    model_size = (ws.abs() >= FREEZE_THR).sum(dim=1).float()
    if len(model_size) >= window:
        rolling_mean = model_size.unfold(0, window, 1).mean(dim=1)
        axes[1].plot(model_size.numpy(), alpha=0.12, color=s["color"])
        axes[1].plot(range(window // 2, len(rolling_mean) + window // 2),
                      rolling_mean.numpy(), color=s["color"], ls=s["ls"], lw=1.6, label=s["label"])
    else:
        axes[1].plot(model_size.numpy(), color=s["color"], ls=s["ls"], lw=1.2, label=s["label"])

    print(f"{s['label']}: resampled-chain sparsity={sparsity.mean():.3f}  "
          f"mean active params={model_size.mean():.1f} / {ws.shape[1]}  "
          f"(cold start was {ckpt['achieved_sparsity']:.3f})")

n_active_at_coldstart = int(round(ws.shape[1] * (1 - ckpt["achieved_sparsity"])))
axes[1].axhline(n_active_at_coldstart, color="black", ls=":", lw=1.2,
                 label=f"cold-start active count ({n_active_at_coldstart})")

axes[0].set_xlabel("Parameter rank (sorted by sparsity)")
axes[0].set_ylabel("P(|param| < threshold)")
axes[0].set_title(f"{DATASET.capitalize()} — per-parameter sparsity (sorted)")
axes[0].legend(fontsize=8)

axes[1].set_xlabel("Resampled draw index")
axes[1].set_ylabel("# active params")
axes[1].set_title(f"{DATASET.capitalize()} — model size over resampled draws")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

## 5. Sparsity dynamics — does the cold start hold?

The central question this ablation script exists to answer. Adapted from
`grid_uci_investigate_skeletons.ipynb` section 7d, but re-framed around the cold-start value:
`diag_df["sparsity"]` (`n_frozen / D` per sampler-loop iteration, from the raw diagnostics log,
finer-grained than the resampled `samples` above) is plotted against the `achieved_sparsity`
the pruning stage produced -- a flat trajectory near that line means the sticky dynamics are
holding the MAP-informed structure; a trajectory draining toward 0 means `kappa`
(`prior_inclusion_weight`) is too permissive and the coordinates are thawing faster than they
refreeze.

In [ ]:
available = [stem for stem in sampler_order if runs[stem]["diag_df"] is not None]

if not available:
    print("No runs have diagnostics saved -- nothing to plot here.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

    for stem in available:
        sk = runs[stem]
        df = sk["diag_df"]

        axes[0].plot(df["sparsity"].to_numpy(), color=sk["color"], ls=sk["ls"],
                     lw=1.3, alpha=0.9, label=sk["label"])

        freeze_idx = df.index[df["event_type"] == "freeze"].to_numpy()
        thaw_idx   = df.index[df["event_type"] == "thaw"].to_numpy()
        axes[1].scatter(freeze_idx, np.full(len(freeze_idx), 1.0), marker="|", color=sk["color"],
                         alpha=0.5, s=200, label=f"{sk['label']} freeze")
        axes[1].scatter(thaw_idx, np.full(len(thaw_idx), 0.0), marker="|", color=sk["color"],
                         alpha=0.5, s=200, label=f"{sk['label']} thaw")

    axes[0].axhline(ckpt["achieved_sparsity"], color="black", ls=":", lw=1.3,
                     label=f"cold-start ({ckpt['achieved_sparsity']:.3f})")
    axes[0].set_xlabel("Loop iteration")
    axes[0].set_ylabel("Sparsity (n_frozen / D)")
    axes[0].set_ylim(0, 1)
    axes[0].set_title(f"{DATASET.capitalize()} — sparsity trajectory vs. cold start")
    axes[0].legend(fontsize=8)

    axes[1].set_xlabel("Loop iteration")
    axes[1].set_yticks([0, 1])
    axes[1].set_yticklabels(["thaw", "freeze"])
    axes[1].set_title(f"{DATASET.capitalize()} — freeze/thaw event timing")
    axes[1].legend(fontsize=7, loc="center left", bbox_to_anchor=(1.0, 0.5))

    plt.tight_layout()
    plt.show()

    for stem in available:
        df = runs[stem]["diag_df"]
        n_freeze = (df["event_type"] == "freeze").sum()
        n_thaw   = (df["event_type"] == "thaw").sum()
        final_sp = df["sparsity"].iloc[-1]
        delta = final_sp - ckpt["achieved_sparsity"]
        verdict = "held/grew" if delta >= -0.05 else ("mild drain" if delta >= -0.2 else "drained")
        print(f"{runs[stem]['label']}: {n_freeze} freezes, {n_thaw} thaws, "
              f"final sparsity={final_sp:.3f}  (cold start {ckpt['achieved_sparsity']:.3f}, "
              f"delta={delta:+.3f} -- {verdict})")

## 6. Mechanism diagnostics

Condensed from `grid_uci_investigate_skeletons.ipynb` sections 7a-7c -- `t_max` (adaptive grid
horizon), event-type composition, and thinning-bound tightness (`max_ratio`). Secondary to
section 5 above, but useful for spotting whether one sampler is paying unusual overhead (e.g. a
high `no_event` share, or `t_max` collapsed to its floor) that would confound a sparsity
comparison between the two families.

In [ ]:
EVENT_TYPES  = ["bounce", "no_event", "freeze", "thaw", "refresh"]
EVENT_COLORS = {"bounce": "#0CCA38", "no_event": "#AAAAAA", "freeze": "#1F77B4",
                 "thaw": "#FA5C00", "refresh": "#9467BD"}

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

# t_max
for stem in available:
    t_max = np.asarray(runs[stem]["payload"]["grid_t_max_log"])
    axes[0].plot(t_max, color=runs[stem]["color"], ls=runs[stem]["ls"], lw=1.1,
                 alpha=0.85, label=runs[stem]["label"])
axes[0].set_yscale("log")
axes[0].set_xlabel("Loop-iteration index (saved tail)")
axes[0].set_ylabel(r"$t_{max}$ (grid horizon)")
axes[0].set_title("Grid horizon")
axes[0].legend(fontsize=8)

# event-type composition
x = np.arange(len(available))
bottom = np.zeros(len(available))
for etype in EVENT_TYPES:
    fracs = np.array([(runs[s]["diag_df"]["event_type"] == etype).mean() for s in available])
    axes[1].bar(x, fracs, bottom=bottom, color=EVENT_COLORS[etype], label=etype, width=0.6)
    bottom += fracs
axes[1].set_xticks(x)
axes[1].set_xticklabels([runs[s]["label"] for s in available], rotation=15, ha="right")
axes[1].set_ylabel("Fraction of loop iterations")
axes[1].set_title("Event-type composition")
axes[1].legend(fontsize=7, loc="upper right", bbox_to_anchor=(1.4, 1.0))

# max_ratio
for stem in available:
    mr = runs[stem]["diag_df"]["max_ratio"].to_numpy()
    axes[2].hist(mr, bins=40, range=(0, 1), density=True, alpha=0.4,
                 color=runs[stem]["color"], histtype="stepfilled", label=runs[stem]["label"])
axes[2].set_xlabel("max_ratio (realized rate / bound)")
axes[2].set_ylabel("Density")
axes[2].set_title("Thinning-bound tightness")
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

for stem in available:
    t_max = np.asarray(runs[stem]["payload"]["grid_t_max_log"])
    mr = runs[stem]["diag_df"]["max_ratio"]
    print(f"{runs[stem]['label']}: t_max mean={t_max.mean():.4g}  "
          f"max_ratio mean={mr.mean():.4f}  bound_violations={runs[stem]['payload']['bound_violations']}")

## 7. Posterior noise

Every run learns noise (Gaussian likelihood, `bm.learns_noise=True`), same as the source
notebook's section 6.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for stem, s in runs.items():
    ns = s["noise_samples"].numpy()
    ax.hist(ns, bins=60, density=True, alpha=0.45, color=s["color"], histtype="stepfilled", edgecolor="none")
    ax.axvline(ns.mean(), color=s["color"], lw=1.8, ls=s["ls"], label=f"{s['label']} (mean={ns.mean():.3f})")
ax.set_xlabel(r"$\sigma$ (standardised scale)")
ax.set_ylabel("Density")
ax.set_title(f"{DATASET.capitalize()} — posterior noise")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()